# Predict Notebook for Time Measurement

## Setup Instructions

**Important Steps:**
1. Make sure you're in the `zac2025` directory
2. **Restart Jupyter kernel** if you see import errors: `Kernel → Restart & Run All`
3. All required `__init__.py` files have been created

**Directory structure required:**
```
zac2025/
├── predict.py
├── models/
│   ├── __init__.py
│   ├── model_loader.py
│   └── tracker.py
└── utils/
    ├── __init__.py
    └── inference_utils.py
```

In [1]:
import os
import torch
import random
import numpy as np
import time
import json
from pathlib import Path

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [2]:
# Add paths and import modules
import sys
import os
from pathlib import Path

# Get the current directory (zac2025 folder)
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")

# Clear any duplicate paths and add fresh
paths_to_add = [
    str(current_dir),
    str(current_dir / 'models'),
    str(current_dir / 'utils')
]

for path in paths_to_add:
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

print(f"Python path (first 5): {sys.path[:5]}")

# Force reload modules if already imported
import importlib
module_names = ['utils.inference_utils', 'models.model_loader', 'models.tracker', 'predict']
for mod_name in module_names:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

# Now import
try:
    from predict import HybridPredictor
    print("✓ HybridPredictor imported successfully")
except Exception as e:
    print(f"✗ Import error: {e}")
    print("\nSOLUTION: Please restart the Jupyter kernel (Kernel → Restart)")
    print("   Then run all cells again.")

Current directory: d:\ZALO-AI-2025\zac2025
Python path (first 5): ['d:\\ZALO-AI-2025\\zac2025\\utils', 'd:\\ZALO-AI-2025\\zac2025\\models', 'd:\\ZALO-AI-2025\\zac2025', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\python311.zip', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\\DLLs']
✓ HybridPredictor imported successfully


In [3]:
# Define paths
test_root = Path("/data/samples") if os.path.exists("/data/samples") else Path("data/public_test/samples")
config_path = Path("/code/config/config.yaml") if os.path.exists("/code/config/config.yaml") else Path("config/config.yaml")

print(f"Test data root: {test_root}")
print(f"Config path: {config_path}")

# Get all video directories
video_dirs = sorted([d for d in test_root.iterdir() if d.is_dir()])
print(f"Found {len(video_dirs)} test cases")

Test data root: data\public_test\samples
Config path: config\config.yaml
Found 6 test cases


In [ ]:
# Process videos and measure time
import csv

all_predicted_time = []
all_result = []

print("Starting inference...")

for video_dir in video_dirs:
    video_id = video_dir.name
    video_path = video_dir / "drone_video.mp4"
    object_images_dir = video_dir / "object_images"
    
    if not video_path.exists():
        print(f"Warning: Video not found for {video_id}, skipping...")
        continue
    
    if not object_images_dir.exists():
        print(f"Warning: Object images not found for {video_id}, skipping...")
        continue
    
    # Get reference images (first 3 images)
    ref_images = sorted(list(object_images_dir.glob('*.jpg')) + list(object_images_dir.glob('*.png')))
    if len(ref_images) < 3:
        print(f"Warning: Less than 3 reference images found for {video_id}, skipping...")
        continue
    
    ref_images = [str(img) for img in ref_images[:3]]
    
    print(f"\nProcessing {video_id}...")
    
    try:
        # Start timing
        t1 = time.time()
        
        # Initialize predictor with reference images
        predictor = HybridPredictor(
            reference_images=ref_images,
            config_path=str(config_path)
        )
        
        # Process video
        predictions = predictor.process_video(
            video_path=str(video_path),
            output_path=None,
            visualize=False
        )
        
        # End timing
        t2 = time.time()
        predicted_time = int((t2 - t1) * 1000)  # Convert to milliseconds
        
        # Format result
        video_result = {
            "video_id": video_id,
            "detections": []
        }
        
        if len(predictions) > 0:
            video_result["detections"].append({
                "bboxes": predictions
            })
        
        all_result.append(video_result)
        all_predicted_time.append((video_id, predicted_time))
        
        print(f"✓ {video_id}: {len(predictions)} frames detected in {predicted_time}ms")
        
    except Exception as e:
        print(f"✗ Error processing {video_id}: {e}")
        all_result.append({
            "video_id": video_id,
            "detections": []
        })
        all_predicted_time.append((video_id, 0))

print("\n" + "="*60)
print("Inference complete!")

# Define output paths
output_dir = Path("/result") if os.path.exists("/result") else Path(".")
submission_json = output_dir / "jupyter_submission.json"
time_csv = output_dir / "time_submission.csv"

# Write submission JSON
with open(submission_json, "w") as f:
    json.dump(all_result, f, indent=2)
print(f"Predictions saved to: {submission_json}")

# Write time CSV
with open(time_csv, "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer", "time"])
    for video_id, p_time in all_predicted_time:
        writer.writerow([video_id, "", p_time])
print(f"Time measurements saved to: {time_csv}")

print("="*60)

Starting inference...

Processing BlackBox_0...
Initializing Hybrid Predictor (YOLOv8 + DINOv2 + Tracking)...
Using device: cpu
Loading YOLOv8 model from saved_models/best.pt...
YOLOv8 loaded successfully. Device: cpu
Loading DINOv2 model: facebook/dinov2-small...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


DINOv2 loaded successfully on cpu
Processing reference images...
Reference features computed: 3 images
Hybrid Predictor initialized successfully!
Processing video: data\public_test\samples\BlackBox_0\drone_video.mp4
Total frames: 5443, FPS: 25.0


Processing:  21%|██▏       | 1158/5443 [05:09<19:07,  3.74it/s]  
